# 为微调BERT设定训练选项


In [ ]:
# 运行准备：使用p5lib按示例代码5.1～5.2读取并划分英文影评数据，并按示例代码5.51、5.53～5.54加载BERT和训练DataLoader
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup
from p5lib.ch5 import BertDataset, create_bert_train_loader, load_bert_model, load_imdb_data

sents_train, sents_test, y_train, y_test = load_imdb_data()
model, tokenizer = load_bert_model()
train_loader = create_bert_train_loader(sents_train, y_train, tokenizer)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
max_epoch = 10
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

## 微调BERT模型

In [ ]:
t0 = time.perf_counter()
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0
print(f"Training finished in {total_time/60:.2f} min")

## BERT性能评测

In [ ]:
from sklearn.metrics import classification_report

test_ds = BertDataset(sents_test, [0]*len(sents_test), tokenizer) #推理阶段无真实标签可用
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
model.eval()
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
print(classification_report(y_test, all_preds, digits=3))

## 用示例代码5.49的样本测试微调后的BERT模型

In [ ]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = BertDataset(input_texts, [0]*len(input_texts), tokenizer)
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        scores = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_bert = np.vstack(all_scores)
print(scores_bert)